# Leaderboard Pipeline 03 — Duplicate Groups and Fold Leakage

This notebook combines exact MD5 groups, pHash pairs, and DINO embedding pairs. It measures whether related images were split across validation folds.

Only exact duplicates and manually confirmed DINO duplicates are considered **strict** grouping evidence. Unreviewed pHash/DINO pairs remain candidate evidence.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts"))
DATA_ROOT = REPO_ROOT / "BDC2026"
MODEL_OUTPUT = REPO_ROOT / "outputs_dinov3_lora_384"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
DISPLAY_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MODEL_OUTPUT:", MODEL_OUTPUT)
print("PIPELINE_ROOT:", PIPELINE_ROOT)


In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

from error_analysis import load_oof, enrich_predictions

STAGE_DIR = PIPELINE_ROOT / "03_duplicates"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
BASE = REPO_ROOT / "eda_outputs" / "notebook_pipeline"

def read_first(paths):
    for path in paths:
        if path.exists():
            print("Loading", path)
            return pd.read_csv(path)
    return pd.DataFrame()

exact = read_first([BASE/"02_exact_duplicates"/"exact_duplicate_groups.csv", REPO_ROOT/"eda_outputs"/"exact_duplicate_groups.csv"])
phash = read_first([BASE/"03_phash"/"phash_duplicate_pairs.csv", REPO_ROOT/"eda_outputs"/"phash_duplicate_pairs.csv"])
dino = read_first([BASE/"04_dino_embeddings"/"dino_pairs_for_manual_review.csv", BASE/"04_dino_embeddings"/"dino_duplicate_pairs.csv", REPO_ROOT/"eda_outputs_dino"/"dino_duplicate_pairs.csv"])
print("Exact rows:", len(exact), "pHash pairs:", len(phash), "DINO pairs:", len(dino))


In [ ]:
def key(path_value, class_name=None):
    path = Path(str(path_value))
    for part in path.parts:
        if part in LABEL2ID:
            return f"{part}/{path.name}"
    return f"{class_name}/{path.name}" if class_name in LABEL2ID else path.name

edges = []
if len(exact):
    for group_id, group in exact.groupby("group_id"):
        keys = [key(row["path"], row.get("class_name")) for _,row in group.iterrows()]
        for other in keys[1:]:
            edges.append({"key_a":keys[0],"key_b":other,"method":"exact_md5","score":1.0,"cross_label":bool(group["cross_label"].iloc[0]),"strict":True})
if len(phash):
    for _,row in phash.iterrows():
        edges.append({"key_a":key(row["path_a"],row.get("class_a")),"key_b":key(row["path_b"],row.get("class_b")),"method":"phash","score":row.get("phash_distance",np.nan),"cross_label":bool(row.get("cross_label",False)),"strict":False})
if len(dino):
    for _,row in dino.iterrows():
        decision = str(row.get("review_decision","")).strip()
        confirmed = decision in {"same_file_or_transform","same_scene_different_frame","confirmed_duplicate"}
        edges.append({"key_a":key(row["path_a"],row.get("class_a")),"key_b":key(row["path_b"],row.get("class_b")),"method":"dino_embedding","score":row.get("cosine_similarity",np.nan),"cross_label":bool(row.get("cross_label",False)),"strict":confirmed})
edges = pd.DataFrame(edges)
display(edges.groupby("method").agg(pairs=("key_a","size"),cross_label=("cross_label","sum"),strict_edges=("strict","sum")))


In [ ]:
class UnionFind:
    def __init__(self): self.parent={}; self.rank={}
    def add(self,x):
        if x not in self.parent: self.parent[x]=x; self.rank[x]=0
    def find(self,x):
        self.add(x)
        if self.parent[x] != x: self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb:return
        if self.rank[ra] < self.rank[rb]: self.parent[ra]=rb
        elif self.rank[ra] > self.rank[rb]: self.parent[rb]=ra
        else: self.parent[rb]=ra; self.rank[ra]+=1

def build_group_map(all_keys, edge_df):
    uf=UnionFind()
    for value in all_keys: uf.add(value)
    for _,row in edge_df.iterrows(): uf.union(row["key_a"],row["key_b"])
    roots={value:uf.find(value) for value in all_keys}
    unique={root:f"G{i:06d}" for i,root in enumerate(sorted(set(roots.values())))}
    return {value:unique[root] for value,root in roots.items()}


In [ ]:
oof = enrich_predictions(load_oof(MODEL_OUTPUT, DATA_ROOT))
oof["path_key"] = [key(path,cls) for path,cls in zip(oof["resolved_path"],oof["class_name"])]
all_keys = set(oof["path_key"])
if len(edges): all_keys.update(edges["key_a"]); all_keys.update(edges["key_b"])
strict_map = build_group_map(all_keys, edges[edges["strict"]] if len(edges) else edges)
candidate_map = build_group_map(all_keys, edges)
oof["strict_group_id"] = oof["path_key"].map(strict_map)
oof["candidate_group_id"] = oof["path_key"].map(candidate_map)


In [ ]:
def leakage_table(group_column):
    rows=[]
    for gid,group in oof.groupby(group_column):
        if len(group)<=1: continue
        rows.append({
            "group_id":gid,"images":len(group),"num_folds":group["fold"].nunique(),
            "folds":",".join(map(str,sorted(group["fold"].unique()))),
            "num_labels":group["label"].nunique(),"labels":",".join(map(str,sorted(group["label"].unique()))),
            "cross_fold_leakage":group["fold"].nunique()>1,
            "cross_label":group["label"].nunique()>1,
            "oof_error_rate":1-group["correct"].mean(),
            "paths":" | ".join(group["resolved_path"].astype(str)),
        })
    return pd.DataFrame(rows)

strict_leakage = leakage_table("strict_group_id")
candidate_leakage = leakage_table("candidate_group_id")
print("Strict duplicate groups:", len(strict_leakage))
print("Strict groups crossing folds:", int(strict_leakage["cross_fold_leakage"].sum()) if len(strict_leakage) else 0)
print("Strict cross-label groups:", int(strict_leakage["cross_label"].sum()) if len(strict_leakage) else 0)
display(strict_leakage.sort_values(["cross_label","cross_fold_leakage","images"], ascending=[False,False,False]).head(30))


In [ ]:
edges.to_csv(STAGE_DIR / "unified_duplicate_edges.csv", index=False)
oof.to_csv(STAGE_DIR / "oof_with_duplicate_groups.csv", index=False)
strict_leakage.to_csv(STAGE_DIR / "strict_group_leakage.csv", index=False)
candidate_leakage.to_csv(STAGE_DIR / "candidate_group_leakage.csv", index=False)
summary = {
    "exact_rows":len(exact),"phash_pairs":len(phash),"dino_pairs":len(dino),
    "strict_groups":len(strict_leakage),
    "strict_groups_crossing_folds":int(strict_leakage["cross_fold_leakage"].sum()) if len(strict_leakage) else 0,
    "strict_cross_label_groups":int(strict_leakage["cross_label"].sum()) if len(strict_leakage) else 0,
    "candidate_groups":len(candidate_leakage),
}
(STAGE_DIR / "duplicate_leakage_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
summary


## Interpretation

- Exact MD5 groups that cross folds indicate validation leakage.
- Cross-label exact groups require annotation review.
- pHash and unreviewed DINO pairs are not automatically safe to group or delete.
- After manual review, mark DINO pairs as `same_file_or_transform` or `same_scene_different_frame`; rerun this notebook so they enter strict groups.

The next implementation stage will use strict groups with `StratifiedGroupKFold`, build a cleaned dataset, and generate controlled training experiments for S02.
